<a href="https://colab.research.google.com/github/Acacia21-code/FlyRankAI-ML-Week1/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Acacia21-code/FlyRankAI-ML-Week1/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [6]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [7]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.544     0.331     0.411      9389
           1      0.683     0.839     0.753     16162

    accuracy                          0.652     25551
   macro avg      0.613     0.585     0.582     25551
weighted avg      0.632     0.652     0.627     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [8]:
con.sql(f"SELECT * FROM {TABLES['dim_content']} LIMIT 5").df()

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


In [9]:
ctr_table = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date > (SELECT MAX(report_date) FROM {TABLES['fact_daily']}) - INTERVAL 90 DAY
    GROUP BY content_hash_id, client_hash_id
    HAVING total_impressions >= 100
""").df()

ctr_table['ctr'] = ctr_table['total_clicks'] / ctr_table['total_impressions']
print(f"{len(ctr_table):,} pages with enough traffic")
ctr_table.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

151,539 pages with enough traffic


,content_hash_id,client_hash_id,total_impressions,total_clicks,avg_position,ctr
0,content_25dfa3e39bc37247,client_e547b89c05043229,1276.0,9.0,8.642429,0.007053
1,content_4f6ed7741dfd65e4,client_e547b89c05043229,107.0,0.0,20.151697,0.000000
2,content_1557a3abbc832229,client_e547b89c05043229,500.0,2.0,17.742535,0.004000
3,content_e1f6d0c859ba9dc4,client_e547b89c05043229,749.0,0.0,23.821131,0.000000
4,content_48537762b74f5b34,client_e547b89c05043229,751.0,0.0,23.079575,0.000000


In [10]:
position_bucket = con.sql("""
    SELECT
        CASE
            WHEN avg_position <= 1 THEN '1'
            WHEN avg_position <= 2 THEN '2'
            WHEN avg_position <= 3 THEN '3'
            WHEN avg_position <= 6 THEN '4-6'
            WHEN avg_position <= 10 THEN '7-10'
            WHEN avg_position <= 20 THEN '11-20'
            ELSE '20+'
        END AS position_bucket,
        SUM(total_impressions) AS bucket_impressions,
        SUM(total_clicks) AS bucket_clicks,
        SUM(total_clicks) * 1.0 / SUM(total_impressions) AS expected_ctr,
        COUNT(*) AS n_pages
    FROM ctr_table
    GROUP BY position_bucket
    ORDER BY MIN(avg_position)
""").df()

position_bucket

,position_bucket,bucket_impressions,bucket_clicks,expected_ctr,n_pages
0,1,7383.0,21.0,0.002844,14
1,2,1825150.0,304997.0,0.167108,103
2,3,18069225.0,130183.0,0.007205,767
3,4-6,209071649.0,990394.0,0.004737,15567
4,7-10,259660498.0,793745.0,0.003057,35624
5,11-20,138749674.0,462643.0,0.003334,43217
6,20+,134273323.0,244715.0,0.001823,56247


In [11]:
bucket_map = position_bucket.set_index('position_bucket')['expected_ctr'].to_dict()

def get_bucket(pos):
    if pos <= 1: return '1'
    elif pos <= 2: return '2'
    elif pos <= 3: return '3'
    elif pos <= 6: return '4-6'
    elif pos <= 10: return '7-10'
    elif pos <= 20: return '11-20'
    else: return '20+'

ctr_table['position_bucket'] = ctr_table['avg_position'].apply(get_bucket)
ctr_table['expected_ctr'] = ctr_table['position_bucket'].map(bucket_map)
ctr_table['ctr_gap'] = ctr_table['expected_ctr'] - ctr_table['ctr']
ctr_table['opportunity_score'] = ctr_table['ctr_gap'] * ctr_table['total_impressions']

top_opportunities = ctr_table[ctr_table['position_bucket'].isin(['4-6','7-10','11-20','20+'])] \
    .sort_values('opportunity_score', ascending=False)

top_opportunities.head(20)

,content_hash_id,client_hash_id,total_impressions,total_clicks,avg_position,ctr,position_bucket,expected_ctr,ctr_gap,opportunity_score
54572,content_943dc881428182b8,client_8ddc46da5414ffd8,680046.0,914.0,3.498362,0.001344,4-6,0.004737,0.003393,2307.448156
110013,content_acbcc847f8996314,client_62f4a7e64f5e0096,604861.0,762.0,4.253691,0.001260,4-6,0.004737,0.003477,2103.289044
119824,content_62770e1299963fe4,client_73cda7b4e4f265ea,522800.0,578.0,5.104457,0.001106,4-6,0.004737,0.003632,1898.557609
100275,content_21309e9a83c83653,client_e547b89c05043229,593168.0,1021.0,4.977519,0.001721,4-6,0.004737,0.003016,1788.898095
109986,content_f352b7cfd0b2f434,client_62f4a7e64f5e0096,517211.0,668.0,4.011424,0.001292,4-6,0.004737,0.003446,1782.081939
104672,content_77276ad7a26f4905,client_e547b89c05043229,497234.0,757.0,5.828985,0.001522,4-6,0.004737,0.003215,1598.448826
44884,content_33d31496fca9665e,client_73cda7b4e4f265ea,552397.0,124.0,6.182972,0.000224,7-10,0.003057,0.002832,1564.598613
134096,content_599eba17030c875c,client_a80fca3f171ed1de,409812.0,429.0,5.873168,0.001047,4-6,0.004737,0.003690,1512.321781
15366,content_39e19a3ec2d95f9d,client_1a730cb2640a1abf,380340.0,8.0,18.225569,0.000021,11-20,0.003334,0.003313,1260.194970
133028,content_9648c4d1595a0794,client_157ffe4d4a595515,272630.0,77.0,3.651719,0.000282,4-6,0.004737,0.004455,1214.476475


In [12]:
opportunities_with_content = top_opportunities.merge(
    con.sql(f"SELECT content_hash_id, word_count, char_count, content_created_date, content_updated_date, last_optimized_date, content_type FROM {TABLES['dim_content']}").df(),
    on='content_hash_id',
    how='left'
)

print("Never optimized:", opportunities_with_content['last_optimized_date'].isna().sum(), "out of", len(opportunities_with_content))
opportunities_with_content[['content_hash_id','ctr','avg_position','word_count','last_optimized_date','content_updated_date']].head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Never optimized: 107916 out of 150655


,content_hash_id,ctr,avg_position,word_count,last_optimized_date,content_updated_date
0,content_943dc881428182b8,0.001344,3.498362,2902,2026-06-22,2026-06-22
1,content_acbcc847f8996314,0.001260,4.253691,<NA>,NaT,2026-07-03
2,content_62770e1299963fe4,0.001106,5.104457,1685,2026-06-30,2026-06-30
3,content_21309e9a83c83653,0.001721,4.977519,1214,2026-06-26,2026-06-26
4,content_f352b7cfd0b2f434,0.001292,4.011424,<NA>,NaT,2026-07-03
5,content_77276ad7a26f4905,0.001522,5.828985,2674,2026-06-22,2026-06-22
6,content_33d31496fca9665e,0.000224,6.182972,<NA>,2026-07-01,2026-07-01
7,content_599eba17030c875c,0.001047,5.873168,2798,NaT,2026-05-20
8,content_39e19a3ec2d95f9d,0.000021,18.225569,2537,2026-05-25,2026-05-25
9,content_9648c4d1595a0794,0.000282,3.651719,3528,2026-06-23,2026-06-23


In [13]:
# Rate among your actual TOP opportunities (say top 500 by score)
top_500 = opportunities_with_content.sort_values('opportunity_score', ascending=False).head(500)
top_500_never_optimized_rate = top_500['last_optimized_date'].isna().mean()

# Rate across the whole filtered population (your baseline for comparison)
overall_never_optimized_rate = opportunities_with_content['last_optimized_date'].isna().mean()

print(f"Never-optimized rate in top 500 opportunities: {top_500_never_optimized_rate:.1%}")
print(f"Never-optimized rate overall:                  {overall_never_optimized_rate:.1%}")

Never-optimized rate in top 500 opportunities: 37.6%
Never-optimized rate overall:                  71.6%


In [14]:
import pandas as pd
today = pd.Timestamp.now()

opportunities_with_content['days_since_updated'] = (today - pd.to_datetime(opportunities_with_content['content_updated_date'])).dt.days

top_500 = opportunities_with_content.sort_values('opportunity_score', ascending=False).head(500)

print("Median word_count — top 500:", top_500['word_count'].median(), "| overall:", opportunities_with_content['word_count'].median())
print("Median days_since_updated — top 500:", top_500['days_since_updated'].median(), "| overall:", opportunities_with_content['days_since_updated'].median())

Median word_count — top 500: 2696.5 | overall: 2785.0
Median days_since_updated — top 500: 70.5 | overall: 99.0


In [15]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           ANY_VALUE(rare_impressions_share) AS rare_share,
           ANY_VALUE(anonymized_impressions_share) AS anon_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

opportunities_with_content = opportunities_with_content.merge(qsignals, on='content_hash_id', how='left')

top_500 = opportunities_with_content.sort_values('opportunity_score', ascending=False).head(500)

print("Median visible_queries — top 500:", top_500['visible_queries'].median(), "| overall:", opportunities_with_content['visible_queries'].median())
print("Median rare_share — top 500:", top_500['rare_share'].median(), "| overall:", opportunities_with_content['rare_share'].median())

Median visible_queries — top 500: 214.0 | overall: 7.0
Median rare_share — top 500: 0.00894225318073895 | overall: 0.10281147642827496


In [16]:
correlation = opportunities_with_content['visible_queries'].corr(opportunities_with_content['ctr_gap'])
print(f"Correlation between visible_queries and ctr_gap: {correlation:.3f}")

# Also check in quartiles, easier to read/chart
opportunities_with_content['query_count_bucket'] = pd.qcut(opportunities_with_content['visible_queries'].fillna(0), 4, duplicates='drop')
opportunities_with_content.groupby('query_count_bucket')['ctr'].median()

Correlation between visible_queries and ctr_gap: 0.051


/tmp/ipykernel_1260/3201037097.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  opportunities_with_content.groupby('query_count_bucket')['ctr'].median()


,ctr
query_count_bucket,
"(-0.001, 1.0]",0.000000
"(1.0, 5.0]",0.001490
"(5.0, 15.0]",0.001881
"(15.0, 1551.0]",0.002029
